# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come
from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: "Refreshing Pages Actually Works" (Part IV, Refresh ROI).** The paper reports
median impression lift by segment when comparing refreshed vs. stale pages — for example,
pages 365+ days old with HIGH competition show a reported median lift of +7,400%, and the
paper states "7 of 9 strata show statistically significant refresh lift" with 95% confidence
intervals.

**My methodology question:** where does "refreshed" come from as a label — is it randomly
assigned, or is it whichever pages an editor happened to choose to refresh? If editors tend
to refresh pages that were already showing early signs of recovering (or that had other
favorable signals), the lift could partly reflect *selection* rather than the refresh itself
causing the lift. The paper's own Method & Limitations section says "this is a pattern study
— we found patterns, not proof of cause and effect," which is the right caveat, but the
Playbook section elsewhere states results more like recommendations ("refresh mature pages
before they decay ... this was the single strongest signal") without repeating that caveat.
My question: was refresh timing itself independent of a page's prior trajectory, and is that
checked anywhere?

**Finding 2: "Which Pages Will Grow?" (Part IV, Growth Prediction model).** The paper reports
a model trained on 96,600 pages "that were clearly growing or declining," reaching 90%
accuracy on same-brand held-out pages and 75% on entirely unseen brands.

**My methodology question:** the phrase "clearly growing or declining" suggests borderline or
ambiguous pages were excluded from training and evaluation. If the model is only tested on
pages that already show a clear trend, that likely inflates the accuracy number relative to
what you'd see scoring the full, messier population a reviewer actually faces day to day
(including flat, new, or ambiguous pages). Was accuracy also measured on the full unfiltered
population, and if so, what did that number look like?

**Framing note:** these questions are asked the way I'd want my own work reviewed — the paper
does state its own limitations clearly in the Method section, and I'm applying that same
standard to my own model in the rest of this notebook below.

In [ ]:
print("Paper findings audited (see markdown above):")
print("1. Refresh ROI (Part IV) - selection bias question: is refresh timing independent of prior trend?")
print("2. Growth Prediction model (Part IV) - 'clearly growing/declining' filter may inflate accuracy")
print("No code needed for this section - this is a methodology reading exercise, same spirit as the")
print("live session's paper walkthrough (docs/flyrank-seo-research-march-2026.pdf).")

Paper findings audited (see markdown above):
1. Refresh ROI (Part IV) - selection bias question: is refresh timing independent of prior trend?
2. Growth Prediction model (Part IV) - 'clearly growing/declining' filter may inflate accuracy
No code needed for this section - this is a methodology reading exercise, same spirit as the
live session's paper walkthrough (docs/flyrank-seo-research-march-2026.pdf).


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** a naive random row-level split — the kind of split someone might reach for
without thinking about it, where pages from the same client can land in both train and test.

**After:** the client-grouped split I actually used in ML-08 — no client's pages appear in
both train and test.

If "before" scores noticeably higher, that gap is inflated performance coming from the model
partly recognizing a client it already saw in training, not from genuinely predicting decline
on unseen clients.

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from huggingface_hub import login, notebook_login
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    notebook_login()
    HF_TOKEN = os.environ.get("HF_TOKEN")

data_files = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
dataset = load_dataset("parquet", data_files=data_files, token=HF_TOKEN)
df_slice = dataset["train"].to_pandas()
df_slice["report_date"] = pd.to_datetime(df_slice["report_date"])

page = df_slice.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_with_data=("report_date", "nunique"),
).reset_index()
page["ctr"] = np.where(page["impressions"] > 0, page["clicks"] / page["impressions"], 0)

median_date = df_slice["report_date"].median()
first_clicks = df_slice[df_slice["report_date"] < median_date].groupby("content_hash_id")["gsc_clicks"].sum()
second_clicks = df_slice[df_slice["report_date"] >= median_date].groupby("content_hash_id")["gsc_clicks"].sum()
page["first_half_clicks"] = page["content_hash_id"].map(first_clicks).fillna(0)
page["second_half_clicks"] = page["content_hash_id"].map(second_clicks).fillna(0)
page["is_declining_label"] = (page["second_half_clicks"] < page["first_half_clicks"]).astype(int)

RANDOM_STATE = 42
feature_cols = ["impressions", "avg_position", "ctr", "days_with_data"]  # clicks removed
X = page[feature_cols].fillna(0)
y = page["is_declining_label"]

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
model_naive = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                      min_samples_leaf=25, n_estimators=200,
                                      random_state=RANDOM_STATE, n_jobs=-1)
model_naive.fit(X_train_naive, y_train_naive)
naive_auc = roc_auc_score(y_test_naive, model_naive.predict_proba(X_test_naive)[:, 1])

clients = page["client_hash_id"].unique()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(clients)
test_clients = set(shuffled[: max(1, int(len(shuffled) * 0.2))])
test_mask = page["client_hash_id"].isin(test_clients)

X_train_grp, X_test_grp = X[~test_mask], X[test_mask]
y_train_grp, y_test_grp = y[~test_mask], y[test_mask]
model_grouped = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                        min_samples_leaf=25, n_estimators=200,
                                        random_state=RANDOM_STATE, n_jobs=-1)
model_grouped.fit(X_train_grp, y_train_grp)
grouped_auc = roc_auc_score(y_test_grp, model_grouped.predict_proba(X_test_grp)[:, 1])

print(f"BEFORE (naive random split)   ROC-AUC: {naive_auc:.4f}")
print(f"AFTER  (client-grouped split) ROC-AUC: {grouped_auc:.4f}")
print(f"Gap: {naive_auc - grouped_auc:+.4f}")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

BEFORE (naive random split)   ROC-AUC: 0.9481
AFTER  (client-grouped split) ROC-AUC: 0.9124
Gap: +0.0357


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Features used: `impressions`, `clicks`, `avg_position`, `ctr`, `days_with_data` — all
whole-month aggregates. `first_half_clicks` and `second_half_clicks` only build the label
and are deliberately excluded from the feature set. The deliberate-leak trap below adds
`second_half_clicks` back in as a feature, to watch the score jump toward perfect, then
removes it and keeps the honest number.

In [ ]:
leaky_features = feature_cols + ["second_half_clicks"]
X_leaky = page[leaky_features].fillna(0)

X_train_leaky, X_test_leaky = X_leaky[~test_mask], X_leaky[test_mask]
model_leaky = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                      min_samples_leaf=25, n_estimators=200,
                                      random_state=RANDOM_STATE, n_jobs=-1)
model_leaky.fit(X_train_leaky, y_train_grp)
leaky_auc = roc_auc_score(y_test_grp, model_leaky.predict_proba(X_test_leaky)[:, 1])

print(f"HONEST features ROC-AUC: {grouped_auc:.4f}")
print(f"LEAKY features (with second_half_clicks) ROC-AUC: {leaky_auc:.4f}")
print(f"Jump from the leak: {leaky_auc - grouped_auc:+.4f}")
print("\nConfirmed: final feature set excludes second_half_clicks, first_half_clicks, AND clicks")
print("(clicks removed after the ML-08 review — its whole-month total sums the two label halves).")
print("Final features used:", feature_cols)

HONEST features ROC-AUC: 0.9124
LEAKY features (with second_half_clicks) ROC-AUC: 0.9849
Jump from the leak: +0.0725

Confirmed: final feature set excludes second_half_clicks, first_half_clicks, AND clicks
(clicks removed after the ML-08 review — its whole-month total sums the two label halves).
Final features used: ['impressions', 'avg_position', 'ctr', 'days_with_data']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured,
directional, decision-support.*

**Original (too bold):** "The model predicts which pages will decline."

**Rewritten (safe):** "On this month's warehouse slice, using a client-holdout split, the
model's ranked output showed a measured Precision@50 of *(fill in from ML-08 results_df)*
compared to a baseline rule's *(fill in)* — a directional signal for prioritizing review, not
a guarantee for any individual page, and not a causal claim about why a page is declining."

**Original (too bold):** "This page is declining because it's stale."

**Rewritten (safe):** "This page's decline is observed alongside low consistency and demand
signals; the association is directional and decision-support only — refreshing it may or may
not cause recovery, which this data cannot establish without an experiment."

In [ ]:
print("Claim rewrite complete (see markdown above).")
print("Reminder for the capstone report: every public claim should use observed / measured /")
print("directional / decision-support language, and avoid 'predicted Google's algorithm' or")
print("any causal wording without an experimental design.")

Claim rewrite complete (see markdown above).
Reminder for the capstone report: every public claim should use observed / measured /
directional / decision-support language, and avoid 'predicted Google's algorithm' or
any causal wording without an experimental design.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.